# 04 · Snake masks with Grounding DINO and SAM 2.1
Makes a pixel mask of the snake in each training photo, to supervise the segmentation head. Neither model runs on the phone. Run with **GPU T4 x2** and internet on. `SHARD_INDEX` and `SHARD_COUNT` let several accounts share the work.

1. Grounding DINO runs twice, for "a snake." and for "a hand. a glove.", because one combined prompt tagged some snakes as hands.
2. SAM 2.1 masks the boxes. We keep the best snake box, plus a second only if it overlaps and scores nearly as high, then cut out hands and gloves.
3. Doubtful masks are flagged for a hand check.

We use SAM 2.1 because SAM 3's weights are gated.

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import json, threading, time, zipfile
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
import torch
import transformers
from PIL import Image, ImageDraw
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, Sam2Model, Sam2Processor

MODE = "pilot"                   # "pilot" (500 photos), "full" (capped per class) or "rest" (every training photo not yet masked)
SHARD_INDEX, SHARD_COUNT = 0, 1
PILOT_PER_CORE_SPECIES = 10
PILOT_PER_WHO_SPECIES = 10
PILOT_TOTAL = 500
FULL_PER_CLASS = 200
SEED = 7

RAW_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes"
SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
MASKS_HANDLE = "puneetbajajiitm/serpentguard-sam-masks"  # read only in "rest" mode
DETECTOR_ID = "IDEA-Research/grounding-dino-tiny"
SAM_ID = "facebook/sam2.1-hiera-large"
SNAKE_PROMPT = "a snake."
HAND_PROMPT = "a hand. a glove."
BOX_THRESHOLD = 0.30
TEXT_THRESHOLD = 0.25
REMOVE_THRESHOLD = 0.45
MAX_HAND_SNAKE_IOU = 0.5
MAX_SNAKE_INSIDE_HAND = 0.8
SECOND_BOX_SCORE_RATIO = 0.9
DETECT_BATCH = 2
DETECTOR_SIZE = {"shortest_edge": 640, "longest_edge": 1066}
PREFETCH = 32
USE_FP16 = True

FLAG_MIN_SCORE = 0.40
FLAG_MIN_FILL, FLAG_MAX_FILL = 0.10, 0.95
FLAG_MIN_AREA, FLAG_MAX_AREA = 0.01, 0.60

OVERLAY_SIZE, GRID_COLUMNS, GRID_ROWS = 256, 10, 5
OUT = Path("/kaggle/working/sam_masks")
for sub in ("masks", "overlays"):
    (OUT / sub).mkdir(parents=True, exist_ok=True)

devices = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
print("devices:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
      "| transformers", transformers.__version__, "| shard", SHARD_INDEX, "of", SHARD_COUNT)

## Choose photos
Training photos only. `pilot` is a fixed 500-photo sample, `full` takes up to 200 per class, and `rest` takes every photo not yet masked. Shards split by photo id, so they never overlap.

In [ ]:
raw_root = Path(kagglehub.dataset_download(RAW_HANDLE))
splits_root = Path(kagglehub.dataset_download(SPLITS_HANDLE))
splits = pd.read_csv(splits_root / "splits.csv", dtype={"obs_id": str, "photo_id": str, "observer_id": str})
train = splits[splits["split"] == "train"]
rng = np.random.default_rng(SEED)


def sample(frame, n):
    return frame.iloc[rng.choice(len(frame), size=min(n, len(frame)), replace=False)]


if MODE == "pilot":
    core = train[train["m1_core27"]]
    who_extra = train[(train["who_category"] != "not_listed") & ~train["m1_core27"]]
    picked = pd.concat([sample(g, PILOT_PER_CORE_SPECIES) for _, g in core.groupby("species")]
                       + [sample(g, PILOT_PER_WHO_SPECIES) for _, g in who_extra.groupby("species")])
    rest = train[~train.index.isin(picked.index)]
    picked = pd.concat([picked, sample(rest, max(PILOT_TOTAL - len(picked), 0))])
elif MODE == "rest":
    done = pd.read_csv(Path(kagglehub.dataset_download(MASKS_HANDLE)) / "mask_stats.csv", usecols=["path"])
    picked = train[~train["path"].isin(done["path"])]
    print(len(done), "photos already masked,", len(picked), "training photos left")
else:
    picked = pd.concat([sample(g, FULL_PER_CLASS) for _, g in train.groupby("species")])
picked = picked.reset_index(drop=True)
picked = picked[picked["photo_id"].astype("int64") % SHARD_COUNT == SHARD_INDEX].reset_index(drop=True)
print(len(picked), "photos in this shard from", picked["species"].nunique(), "species")


def read_photo(file):
    for candidate in (raw_root / "images" / file, raw_root / file):
        if candidate.exists():
            return Image.open(candidate).convert("RGB")
    raise FileNotFoundError(file)

## Segment on every GPU
Each GPU gets its own models and half of the photos.

In [ ]:
def box_iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def contained_fraction(inner, outer):
    ix = max(0.0, min(inner[2], outer[2]) - max(inner[0], outer[0]))
    iy = max(0.0, min(inner[3], outer[3]) - max(inner[1], outer[1]))
    area = (inner[2] - inner[0]) * (inner[3] - inner[1])
    return ix * iy / area if area > 0 else 0.0


def choose_boxes(snake_result, hand_result):
    snakes = sorted(zip(snake_result["scores"].tolist(), snake_result["boxes"].tolist()), key=lambda t: t[0], reverse=True)
    if not snakes:
        return [], [], 0.0
    top = snakes[0][1]
    removes = [b for s, b in zip(hand_result["scores"].tolist(), hand_result["boxes"].tolist())
               if s >= REMOVE_THRESHOLD and box_iou(b, top) < MAX_HAND_SNAKE_IOU
               and contained_fraction(top, b) < MAX_SNAKE_INSIDE_HAND]
    keep = [top]
    for score, box in snakes[1:]:
        if score >= SECOND_BOX_SCORE_RATIO * snakes[0][0] and box_iou(box, snakes[0][1]) > 0:
            keep.append(box)
            break
    return keep, removes, snakes[0][0]


def overlay(image, mask, boxes, removed):
    tinted = np.asarray(image, dtype=np.float32)
    tinted[mask] = 0.55 * tinted[mask] + 0.45 * np.array([235, 104, 52], dtype=np.float32)
    canvas = Image.fromarray(tinted.astype(np.uint8))
    draw = ImageDraw.Draw(canvas)
    for box in boxes:
        draw.rectangle(box, outline=(42, 120, 214), width=3)
    for box in removed:
        draw.rectangle(box, outline=(255, 255, 255), width=3)
    return canvas.resize((OVERLAY_SIZE, OVERLAY_SIZE))


class Worker:
    def __init__(self, device):
        self.device = device
        self.dtype = torch.float16 if (USE_FP16 and device.startswith("cuda")) else torch.float32
        self.det_processor = AutoProcessor.from_pretrained(DETECTOR_ID)
        self.det_processor.image_processor.size = DETECTOR_SIZE
        self.detector = AutoModelForZeroShotObjectDetection.from_pretrained(DETECTOR_ID).to(device).eval()
        self.sam_processor = Sam2Processor.from_pretrained(SAM_ID)
        self.sam = Sam2Model.from_pretrained(SAM_ID).to(device).eval()

    @torch.inference_mode()
    def detect(self, images, prompt):
        inputs = self.det_processor(images=images, text=[prompt] * len(images), return_tensors="pt").to(self.device)
        with torch.autocast("cuda", dtype=self.dtype, enabled=self.dtype == torch.float16):
            outputs = self.detector(**inputs)
        return self.det_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
            target_sizes=[im.size[::-1] for im in images])

    @torch.inference_mode()
    def segment(self, image, boxes):
        inputs = self.sam_processor(images=image, input_boxes=[boxes], return_tensors="pt").to(self.device)
        with torch.autocast("cuda", dtype=self.dtype, enabled=self.dtype == torch.float16):
            outputs = self.sam(**inputs, multimask_output=False)
        masks = self.sam_processor.post_process_masks(outputs.pred_masks.float().cpu(), inputs["original_sizes"])[0]
        return masks.reshape(-1, *masks.shape[-2:]).numpy(), outputs.iou_scores.float().cpu().flatten().tolist()

    def run(self, rows, results, lock):
        with ThreadPoolExecutor(4) as reader:
            for start in range(0, len(rows), DETECT_BATCH):
                batch = rows[start:start + DETECT_BATCH]
                images = list(reader.map(lambda r: read_photo(r["file"]), batch))
                snake_results, hand_results = self.detect(images, SNAKE_PROMPT), self.detect(images, HAND_PROMPT)
                for row, image, snake_result, hand_result in zip(batch, images, snake_results, hand_results):
                    results_row = self.one(row, image, snake_result, hand_result)
                    with lock:
                        results.append(results_row)
                        if len(results) % 1000 == 0:
                            print(len(results), "photos done", flush=True)

    def one(self, row, image, snake_result, hand_result):
        t0 = time.time()
        snake_boxes, remove_boxes, score = choose_boxes(snake_result, hand_result)
        record = {"path": row["path"], "photo_id": row["photo_id"], "species": row["species"],
                  "who_category": row["who_category"], "photo_position": row["photo_position"],
                  "n_snake_boxes": len(snake_boxes), "n_removed_boxes": len(remove_boxes), "det_score": score}
        if snake_boxes:
            masks, sam_scores = self.segment(image, snake_boxes + remove_boxes)
            snake = masks[:len(snake_boxes)].any(axis=0)
            if remove_boxes:
                snake &= ~masks[len(snake_boxes):].any(axis=0)
            x0, y0, x1, y1 = snake_boxes[0]
            box_area = max((x1 - x0) * (y1 - y0), 1.0)
            Image.fromarray((snake * 255).astype(np.uint8)).save(OUT / "masks" / f"{Path(row['path']).stem}.png")
            record.update(mask_area_fraction=float(snake.mean()), fill_ratio=float(snake.sum() / box_area),
                          min_sam_score=min(sam_scores[:len(snake_boxes)]), snake_boxes=snake_boxes, removed_boxes=remove_boxes)
            overlay(image, snake, snake_boxes, remove_boxes).save(OUT / "overlays" / f"{Path(row['path']).stem}.jpg", quality=85)
        record["seconds"] = time.time() - t0
        return record


workers = [Worker(device) for device in devices]
rows = picked.to_dict("records")
results, errors, lock, started = [], [], threading.Lock(), time.time()


def guarded(worker, part):
    try:
        worker.run(part, results, lock)
    except Exception as error:
        errors.append(error)
        raise


threads = [threading.Thread(target=guarded, args=(w, rows[i::len(workers)])) for i, w in enumerate(workers)]
for t in threads:
    t.start()
for t in threads:
    t.join()
if errors:
    raise errors[0]
elapsed = time.time() - started
print(f"{len(results)} photos in {elapsed / 60:.1f} min, {elapsed / max(len(results), 1):.2f} s per photo overall")

## Quality flags and review sheets
A mask is flagged when the detector score is low, or the mask fills too little or too much of its box or the photo. Review sheets show flagged masks first.

In [ ]:
stats = pd.DataFrame(results)
has_mask = stats["n_snake_boxes"] > 0
stats["flagged"] = ~has_mask | (stats["det_score"] < FLAG_MIN_SCORE)
stats.loc[has_mask, "flagged"] |= ~stats.loc[has_mask, "fill_ratio"].between(FLAG_MIN_FILL, FLAG_MAX_FILL)
stats.loc[has_mask, "flagged"] |= ~stats.loc[has_mask, "mask_area_fraction"].between(FLAG_MIN_AREA, FLAG_MAX_AREA)
stats.to_csv(OUT / "mask_stats.csv", index=False)

review_rng = np.random.default_rng(SEED)
unflagged = stats[has_mask & ~stats["flagged"]]["path"].tolist()
review_paths = stats[has_mask & stats["flagged"]]["path"].tolist()
review_paths += list(review_rng.choice(unflagged, size=min(len(unflagged), 200), replace=False))
review = pd.DataFrame({"overlay": [f"{Path(p).stem}.jpg" for p in review_paths], "path": review_paths})
review = review.merge(stats[["path", "species", "flagged", "det_score", "fill_ratio", "mask_area_fraction"]], on="path")
review["mask_ok"] = ""
review.to_csv(OUT / "review.csv", index=False)

per_sheet = GRID_COLUMNS * GRID_ROWS
overlays = [OUT / "overlays" / name for name in review["overlay"]]
for sheet, start in enumerate(range(0, len(overlays), per_sheet)):
    canvas = Image.new("RGB", (GRID_COLUMNS * OVERLAY_SIZE, GRID_ROWS * OVERLAY_SIZE), "white")
    for j, path in enumerate(overlays[start:start + per_sheet]):
        canvas.paste(Image.open(path), ((j % GRID_COLUMNS) * OVERLAY_SIZE, (j // GRID_COLUMNS) * OVERLAY_SIZE))
    canvas.save(OUT / f"review_sheet_{sheet:02d}.jpg", quality=85)

summary = {
    "mode": MODE, "shard": f"{SHARD_INDEX}/{SHARD_COUNT}", "devices": devices, "photos": len(stats),
    "minutes": round(elapsed / 60, 1), "seconds_per_photo_overall": round(elapsed / max(len(stats), 1), 3),
    "no_snake_found": int((~has_mask).sum()), "flagged": int(stats["flagged"].sum()),
    "hand_or_glove_removed": int((stats["n_removed_boxes"] > 0).sum()),
    "two_snake_boxes": int((stats["n_snake_boxes"] > 1).sum()),
    "median_mask_area_fraction": round(float(stats.loc[has_mask, "mask_area_fraction"].median()), 3),
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1))
with zipfile.ZipFile(OUT.parent / f"masks_shard{SHARD_INDEX}.zip", "w") as zf:
    for p in (OUT / "masks").glob("*.png"):
        zf.write(p, p.name)
summary